# 01 — Bank distances

Reduces raw WOCU bank point clouds to a single `dist_m` value per scope region per survey year.

**Algorithm:**
1. Keep only `status == 'OK'` points.
2. Select the `N_POINTS` furthest OK points per region × year.
3. Take their mean distance from the centreline → `dist_m`.

**Input:** `01_raw/erosion/wocu_output_fase2_20260210.gpkg` — layer `punten_oever`

**Output:** `03_features/EXPERIMENT/dist_per_year.parquet` — `(location_id, year, dist_m, n_ok_pts)`

In [1]:
import os, sys
from pathlib import Path

_cwd = Path.cwd()
for candidate in [_cwd, _cwd.parent, _cwd.parent.parent]:
    if (candidate / 'src').exists():
        _backend = candidate
        break
else:
    _backend = _cwd

os.chdir(_backend)
sys.path.insert(0, str(_backend))
print('cwd:', os.getcwd())

cwd: /Users/admin/Documents/work/Moraine/oevererosie/wocu-oevererosie/backend/notebooks/04_model/20260314_pipeline_building


In [2]:
import geopandas as gpd
import pandas as pd

import src.paths as PATHS

# ── Experiment config ─────────────────────────────────────────────────────────
EXPERIMENT   = '20260314'
N_POINTS     = 3       # top-N furthest OK points per region × year

RAW_GPKG     = PATHS.DATA_DIR / '01_raw/erosion/wocu_output_fase2_20260210.gpkg'
FEATURES_DIR = PATHS.DATA_DIR / f'03_features/{EXPERIMENT}'
FEATURES_DIR.mkdir(parents=True, exist_ok=True)

OUT_PARQUET  = FEATURES_DIR / 'dist_per_year.parquet'

print(f'Experiment : {EXPERIMENT}')
print(f'Features → : {FEATURES_DIR}')

Experiment : 20260314
Features → : /Users/admin/Documents/work/Moraine/oevererosie/wocu-oevererosie/backend/data/03_features/20260314


## 1. Load raw bank points

In [3]:
print('Reading punten_oever ...')
pts = gpd.read_file(RAW_GPKG, layer='punten_oever')
print(f'  {len(pts):,} rows  CRS: {pts.crs}')
print(f'  columns: {list(pts.columns)}')
print(f'  status values: {pts["status"].value_counts().to_dict()}')
print(f'  dtm_date years: {sorted(pts["dtm_date"].unique())}')

Reading punten_oever ...
  4,844,655 rows  CRS: EPSG:28992
  columns: ['z', 'dist', 'type_oever', 'status', 'location_id', 'dtm_version', 'dtm_date', 'geometry']
  status values: {'OK': 4078502, 'OUTLIER': 569299, 'UNCERTAIN': 196854}
  dtm_date years: [np.int64(2015), np.int64(2016), np.int64(2017), np.int64(2020), np.int64(2021), np.int64(2022), np.int64(2024), np.int64(2025)]


## 2. Filter to OK points and select top-N per region × year

In [4]:
ok = pts[pts['status'] == 'OK'].copy()
print(f'OK points: {len(ok):,}  ({len(ok)/len(pts)*100:.1f}% of total)')

# Rename dtm_date → year (integer)
ok = ok.rename(columns={'dtm_date': 'year'})
ok['year'] = ok['year'].astype(int)

# Select top-N furthest OK points per (location_id, year)
def top_n_mean_dist(group, n=N_POINTS):
    chosen = group.nlargest(n, 'dist')
    return pd.Series({
        'dist_m':    chosen['dist'].mean(),
        'n_ok_pts':  len(group),           # total OK pts in this region × year
        'n_selected': len(chosen),
    })

dist_per_year = (
    ok.groupby(['location_id', 'year'], group_keys=False)
    .apply(top_n_mean_dist, include_groups=False)
    .reset_index()
)

print(f'\ndist_per_year shape: {dist_per_year.shape}')
print(f'  unique locations : {dist_per_year["location_id"].nunique():,}')
print(f'  years covered    : {sorted(dist_per_year["year"].unique())}')
dist_per_year.head(6)

OK points: 4,078,502  (84.2% of total)

dist_per_year shape: (30228, 5)
  unique locations : 10,484
  years covered    : [np.int64(2015), np.int64(2016), np.int64(2017), np.int64(2020), np.int64(2021), np.int64(2022), np.int64(2024), np.int64(2025)]


,location_id,year,dist_m,n_ok_pts,n_selected
0,ijssel1_l_0000_0010,2017,162.583333,97.0,3.0
1,ijssel1_l_0000_0010,2025,162.083333,110.0,3.0
2,ijssel1_l_0010_0020,2017,155.478645,72.0,3.0
3,ijssel1_l_0010_0020,2022,156.229750,106.0,3.0
4,ijssel1_l_0010_0020,2025,156.730486,87.0,3.0
5,ijssel1_l_0020_0030,2017,168.083333,102.0,3.0


## 3. Sanity checks

In [5]:
# Every dist_m must be positive
assert (dist_per_year['dist_m'] > 0).all(), 'Some dist_m values are non-positive!'

# No duplicate (location_id, year) pairs
dupes = dist_per_year.duplicated(['location_id', 'year']).sum()
assert dupes == 0, f'{dupes} duplicate (location_id, year) pairs!'

# Timestamps per location
ts_counts = dist_per_year.groupby('location_id')['year'].count()
print('Timestamps per location:')
print(ts_counts.value_counts().sort_index().to_string())
print(f'\nTotal: {len(dist_per_year):,} rows across {ts_counts.index.nunique():,} locations')
print('All checks passed.')

Timestamps per location:
year
1     142
2     940
3    9402

Total: 30,228 rows across 10,484 locations
All checks passed.


## 4. Save

In [6]:
dist_per_year.to_parquet(OUT_PARQUET, index=False)
print(f'Saved → {OUT_PARQUET}  ({OUT_PARQUET.stat().st_size / 1e3:.0f} KB)')
print(f'Shape : {dist_per_year.shape}')
print(dist_per_year.dtypes)

Saved → /Users/admin/Documents/work/Moraine/oevererosie/wocu-oevererosie/backend/data/03_features/20260314/dist_per_year.parquet  (432 KB)
Shape : (30228, 5)
location_id     object
year             int64
dist_m         float64
n_ok_pts       float64
n_selected     float64
dtype: object
